# 06 · Prediction：agent state 到未来分布

本章把 Chapter 03 的 tracked state 变成多模态 future trajectories，并合并旧版 `00E` 的 prediction 入口和 `08` 的 ADE/FDE/Miss Rate。Prediction 不是“猜一条最像 label 的线”：cut-in/keep-lane/brake 等多个 mode 都可能合理，planning 需要看分布和风险。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ad_tutorial import (
    ARTIFACT_DIR,
    BEVConfig,
    build_bev_dataset,
    build_urban_cut_in_scene,
    ensure_artifact_dir,
    load_json_artifact,
    load_numpy_artifact,
    save_json_artifact,
    save_numpy_artifact,
    scene_to_bev,
)

ensure_artifact_dir()
print("project root:", PROJECT_ROOT)
print("artifact directory:", ARTIFACT_DIR)

import numpy as np
import matplotlib.pyplot as plt

state = load_numpy_artifact("03_temporal_state.npz")
time_s = state["time_s"]
last_position = state["estimate_xy"][-1]
last_velocity = state["velocity_xy"][-1]
horizon_s = np.arange(0.1, 3.1, 0.1)
true_future = np.array([build_urban_cut_in_scene(seed=7, timestamp_s=float(time_s[-1] + dt)).cut_in_xy for dt in horizon_s])

keep_lane = last_position + np.c_[last_velocity[0] * horizon_s, np.zeros_like(horizon_s)]
cut_in = last_position + np.c_[last_velocity[0] * horizon_s, last_velocity[1] * horizon_s]
brake = last_position + np.c_[0.5 * last_velocity[0] * horizon_s, 0.4 * last_velocity[1] * horizon_s]
candidates = np.stack([keep_lane, cut_in, brake])
probabilities = np.array([0.25, 0.55, 0.20])

In [ ]:
def ade(prediction, truth):
    return float(np.mean(np.linalg.norm(prediction - truth[None, ...], axis=-1)))

def fde(prediction, truth):
    return float(np.mean(np.linalg.norm(prediction[:, -1] - truth[-1], axis=-1)))

per_mode_ade = np.mean(np.linalg.norm(candidates - true_future[None, ...], axis=-1), axis=1)
per_mode_fde = np.linalg.norm(candidates[:, -1] - true_future[-1], axis=-1)
min_ade = float(per_mode_ade.min())
min_fde = float(per_mode_fde.min())
miss_rate = float(np.mean(per_mode_fde > 2.0))
print({"per_mode_ADE": per_mode_ade.round(3).tolist(), "per_mode_FDE": per_mode_fde.round(3).tolist(),
       "minADE": round(min_ade, 3), "minFDE": round(min_fde, 3), "miss_rate@2m": round(miss_rate, 3)})

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(true_future[:, 0], true_future[:, 1], "k-", linewidth=3, label="future")
for name, path, prob in zip(["keep lane", "cut in", "brake"], candidates, probabilities):
    ax.plot(path[:, 0], path[:, 1], label=f"{name} p={prob:.2f}")
ax.scatter(last_position[0], last_position[1], color="red", label="current state")
ax.set_aspect("equal")
ax.legend()
ax.set(title="Multimodal prediction for the tracked cut-in actor", xlabel="x / m", ylabel="y / m")
plt.show()

## Metrics are not interchangeable

`ADE/FDE` summarize geometric distance; `minADE/minFDE` reward covering one plausible mode; `miss rate` exposes tail failure. A planner may care more about a low-probability cut-in mode than a mean displacement. This is why prediction metrics alone cannot substitute closed-loop evaluation.

In [ ]:
mode_weights = np.linspace(0.0, 1.0, 11)
risk_aware_cost = []
for cut_in_weight in mode_weights:
    costs = per_mode_ade + cut_in_weight * np.array([0.0, 2.5, 0.5])
    risk_aware_cost.append(int(np.argmin(costs)))
print("planner-selected mode as cut-in risk weight changes:", list(zip(mode_weights.round(2), risk_aware_cost)))

save_numpy_artifact("06_prediction.npz", horizon_s=horizon_s, truth_future=true_future,
                    candidates=candidates, probabilities=probabilities, per_mode_ade=per_mode_ade,
                    per_mode_fde=per_mode_fde)
save_json_artifact("06_prediction_metrics.json", {
    "minADE": min_ade, "minFDE": min_fde, "miss_rate_at_2m": miss_rate,
    "modes": ["keep_lane", "cut_in", "brake"], "next": "07_planning_closed_loop.ipynb",
})
print("saved prediction artifact")

### 完成标准

解释为什么“更低 ADE”不一定意味着更低 closed-loop collision rate；为 cut-in 设计一个 mode coverage 或 risk-weighted metric；并说明 prediction 的输入为什么应该是 tracked state，而不是未经时间语义处理的单帧 boxes。